# reddit-scraper · Colab Training

Trains two models from your scraped `full_*.json` files:

| Model | Input | Output |
|---|---|---|
| **Sentiment classifier** | comment body text | positive / negative / neutral |
| **Viral predictor** | post title + engagement features | hype score 0.0–1.0 |

**Workflow:**
1. Collect data locally: `python main.py scrape --subreddits MMA ufc boxing --limit 50 --output data/`
2. Upload the `full_*.json` file(s) here
3. Run all cells — models saved to Google Drive
4. Download `sentiment_clf.pkl` + `viral_clf.pkl` → put in `models/`

> Minimum recommended: 20+ posts, 500+ comments. More data = better models.

## 1. Setup

In [ ]:
!pip install -q scikit-learn pandas joblib vaderSentiment

In [ ]:
import json
import re
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
print('Ready.')

## 2. Mount Google Drive (models will be saved here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/reddit-scraper/models')
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f'Models will be saved to: {DRIVE_MODEL_DIR}')

## 3. Load scraped data

Upload one or more `full_*.json` files from your local `output/` folder.

In [ ]:
from google.colab import files

uploaded = files.upload()   # select your full_*.json file(s)
json_paths = list(uploaded.keys())
print(f'Uploaded: {json_paths}')

In [ ]:
# ── parse all uploaded files into flat DataFrames ────────────────────────────

def flatten_comments(comment_list, post_id='', depth=0):
    rows = []
    for c in comment_list:
        body = c.get('body', '').strip()
        if body and body not in ('[deleted]', '[removed]'):
            rows.append({
                'post_id':  post_id,
                'author':   c.get('author', ''),
                'score':    c.get('score', 0),
                'depth':    c.get('depth', depth),
                'body':     body,
                'created':  c.get('created', ''),
            })
        rows.extend(flatten_comments(c.get('replies', []), post_id, depth + 1))
    return rows


all_data   = []
post_rows  = []
cmt_rows   = []

for path in json_paths:
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    all_data.extend(data)

for entry in all_data:
    p = entry.get('post', {})
    post_rows.append({
        'id':           p.get('id', ''),
        'title':        p.get('title', ''),
        'score':        p.get('score', 0),
        'upvote_ratio': p.get('upvote_ratio', 0.5),
        'comment_count':p.get('comment_count', 0),
        'post_type':    p.get('post_type', 'link'),
        'domain':       p.get('domain', ''),
        'subreddit':    p.get('subreddit', ''),
        'created':      p.get('created', ''),
    })
    cmt_rows.extend(flatten_comments(entry.get('comments', []), p.get('id', '')))

df_posts    = pd.DataFrame(post_rows).drop_duplicates('id')
df_comments = pd.DataFrame(cmt_rows)

print(f'Posts:    {len(df_posts)}')
print(f'Comments: {len(df_comments)}')
df_posts.head()

## 4. Exploratory analysis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Post score distribution
axes[0].hist(df_posts['score'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Post score distribution')
axes[0].set_xlabel('score')

# Upvote ratio
axes[1].hist(df_posts['upvote_ratio'], bins=20, color='seagreen', edgecolor='white')
axes[1].set_title('Upvote ratio distribution')
axes[1].set_xlabel('upvote ratio')

# Comment score distribution
axes[2].hist(df_comments['score'].clip(-10, 200), bins=30, color='coral', edgecolor='white')
axes[2].set_title('Comment score distribution')
axes[2].set_xlabel('score (clipped at 200)')

plt.tight_layout()
plt.show()

print('\nPost score stats:')
print(df_posts['score'].describe().round(1))
print('\nComment depth stats:')
print(df_comments['depth'].value_counts().sort_index())

## 5. Model A — Sentiment classifier

Uses VADER compound scores as soft labels to train a TF-IDF + Logistic Regression model.
The ML model learns MMA-specific language patterns that VADER's generic lexicon misses.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

_FIGHT_LEXICON = {
    'insane': 1.5, 'wtf': 0.8, 'goat': 2.5, 'beast': 2.0, 'savage': 1.8,
    'vicious': 1.2, 'nasty': 1.0, 'sick': 1.0, 'filthy': 1.0, 'brutal': 1.0,
    'clean': 1.5, 'robbed': -2.0, 'trash': -1.5, 'boring': -2.0,
    'awful': -2.0, 'pathetic': -2.0, 'disgrace': -2.0, 'juicer': -1.5, 'cheater': -2.0,
}

vader = SentimentIntensityAnalyzer()
vader.lexicon.update(_FIGHT_LEXICON)

def vader_label(text):
    c = vader.polarity_scores(text)['compound']
    if c >= 0.05:  return 'positive'
    if c <= -0.05: return 'negative'
    return 'neutral'

df_comments['sentiment'] = df_comments['body'].apply(vader_label)
df_comments['compound']  = df_comments['body'].apply(
    lambda t: round(vader.polarity_scores(t)['compound'], 3)
)

print('Sentiment distribution (VADER labels used as training targets):')
print(df_comments['sentiment'].value_counts())
df_comments[['body', 'sentiment', 'compound']].sample(5)

In [ ]:
# Filter out very neutral comments (compound near 0) — low signal
df_sent = df_comments[df_comments['compound'].abs() > 0.05].copy()
print(f'Training on {len(df_sent)} comments (removed near-neutral)')

X_text = df_sent['body']
y_sent = df_sent['sentiment']

sentiment_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=15000,
        sublinear_tf=True,
        min_df=2,
        strip_accents='unicode',
    )),
    ('clf', LogisticRegression(
        C=3,
        max_iter=1000,
        class_weight='balanced',
        solver='lbfgs',
        multi_class='multinomial',
    )),
])

# Cross-validation
cv_scores = cross_val_score(sentiment_pipeline, X_text, y_sent, cv=5, scoring='f1_macro')
print(f'\nCV F1-macro: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})')

# Final fit on all data
sentiment_pipeline.fit(X_text, y_sent)

# Quick test
test_phrases = [
    "He's the GOAT, most complete fighter ever",
    "This fight was so boring, terrible decision",
    "Nice technical display but not very exciting",
    "ROBBED!! That judge is blind",
    "Savage KO, beast mode activated",
]
preds = sentiment_pipeline.predict(test_phrases)
print('\nQuick sanity check:')
for phrase, pred in zip(test_phrases, preds):
    print(f'  [{pred:>8}] {phrase}')

## 6. Model B — Viral predictor

Predicts if a post will be viral (top 33% by score) using title text + engagement features.
Only uses features available at or shortly after posting time.

In [ ]:
_HYPE_WORDS = [
    'goat', 'insane', 'iconic', 'legendary', 'greatest', 'breaking',
    'exclusive', 'leaked', 'official', 'ko', 'tko', 'finish', 'shocking',
    'upset', 'champion', 'title', 'belt', 'retirement', 'career',
]

def post_numeric_features(df):
    """Features available right after a post is made."""
    title = df['title'].fillna('').str.lower()
    return pd.DataFrame({
        'upvote_ratio':   df['upvote_ratio'].fillna(0.5),
        'is_self':        (df['post_type'] == 'self').astype(float),
        'is_video':       df['domain'].fillna('').str.contains('youtube|streamable|v.redd|youtu').astype(float),
        'is_image':       df['domain'].fillna('').str.contains('imgur|i.redd|gallery').astype(float),
        'title_len':      df['title'].fillna('').str.len() / 100,
        'has_excl':       df['title'].fillna('').str.contains('!').astype(float),
        'has_question':   df['title'].fillna('').str.contains(r'\?').astype(float),
        'hype_kw_count':  title.apply(lambda t: sum(1 for w in _HYPE_WORDS if w in t)) / len(_HYPE_WORDS),
        'caps_ratio':     df['title'].fillna('').apply(
            lambda t: sum(1 for c in t if c.isupper()) / max(len(t), 1)
        ),
    })

# Label: top 33% by score = viral
threshold = df_posts['score'].quantile(0.67)
df_posts['viral'] = (df_posts['score'] >= threshold).astype(int)

print(f'Viral threshold (top 33% score): {threshold:.0f}')
print(f'Viral posts: {df_posts["viral"].sum()} / {len(df_posts)}')

numeric_df = post_numeric_features(df_posts)
print('\nNumeric feature sample:')
numeric_df.describe().round(3)

In [ ]:
# Build feature matrix: TF-IDF on title + numeric features
title_tfidf_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    sublinear_tf=True,
    strip_accents='unicode',
)

titles   = df_posts['title'].fillna('')
X_title  = title_tfidf_vec.fit_transform(titles)
X_num    = csr_matrix(numeric_df.values.astype(float))
X_viral  = hstack([X_title, X_num])
y_viral  = df_posts['viral'].values

viral_clf = LogisticRegression(
    C=2,
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
)

if len(df_posts) >= 10:
    cv_auc = cross_val_score(viral_clf, X_viral, y_viral, cv=min(5, len(df_posts)//2), scoring='roc_auc')
    print(f'CV ROC-AUC: {cv_auc.mean():.3f} (+/- {cv_auc.std():.3f})')
else:
    print('Warning: fewer than 10 posts — cross-validation skipped. Collect more data for reliable results.')

# Fit on all data
viral_clf.fit(X_viral, y_viral)

# Show predicted hype scores
hype_proba = viral_clf.predict_proba(X_viral)[:, 1]
df_posts['hype_score'] = hype_proba.round(3)

print('\nTop predicted viral posts:')
df_posts[['title', 'score', 'upvote_ratio', 'hype_score']].sort_values('hype_score', ascending=False).head(10)

## 7. Feature importance analysis

In [ ]:
# Top TF-IDF features driving viral prediction
tfidf_feature_names = title_tfidf_vec.get_feature_names_out()
numeric_feature_names = list(numeric_df.columns)
all_feature_names = list(tfidf_feature_names) + numeric_feature_names

coefs = viral_clf.coef_[0]
top_pos = np.argsort(coefs)[::-1][:20]
top_neg = np.argsort(coefs)[:20]

print('Top 20 features BOOSTING viral probability:')
for i in top_pos:
    print(f'  +{coefs[i]:.3f}  {all_feature_names[i]}')

print('\nTop 20 features REDUCING viral probability:')
for i in top_neg:
    print(f'  {coefs[i]:.3f}  {all_feature_names[i]}')

In [ ]:
# Top sentiment words
sent_coefs    = sentiment_pipeline.named_steps['clf'].coef_
sent_features = sentiment_pipeline.named_steps['tfidf'].get_feature_names_out()
sent_classes  = sentiment_pipeline.named_steps['clf'].classes_

print('Sentiment model — top words per class:')
for i, cls in enumerate(sent_classes):
    top_idx = np.argsort(sent_coefs[i])[::-1][:15]
    words   = [sent_features[j] for j in top_idx]
    print(f'\n  [{cls}]  {", ".join(words)}')

## 8. Real-time inference demo

Show how to score new posts without retraining.
This is what the local scraper will do after loading the saved `.pkl` files.

In [ ]:
def score_new_posts(titles_list, upvote_ratios=None, post_types=None, domains=None):
    """
    Score a list of new post titles.
    upvote_ratios, post_types, domains: optional lists aligned with titles_list.
    Returns list of hype scores 0.0-1.0.
    """
    n = len(titles_list)
    df_new = pd.DataFrame({
        'title':        titles_list,
        'upvote_ratio': upvote_ratios if upvote_ratios else [0.85] * n,
        'post_type':    post_types    if post_types    else ['link'] * n,
        'domain':       domains       if domains       else [''] * n,
    })

    X_t = title_tfidf_vec.transform(df_new['title'].fillna(''))
    X_n = csr_matrix(post_numeric_features(df_new).values.astype(float))
    X   = hstack([X_t, X_n])

    return viral_clf.predict_proba(X)[:, 1]


# Test with hypothetical posts
sample_titles = [
    "Jon Jones vs Stipe Miocic OFFICIAL for UFC 310",
    "Weekly discussion thread - what are you training?",
    "LEAKED: Khamzat Chimaev's next opponent revealed",
    "Conor McGregor announces retirement",
    "Cool sparring session video from local gym",
    "Best KO of the year? This finish was INSANE",
]

scores = score_new_posts(sample_titles)

print('Viral hype scores for sample posts:\n')
for title, score in sorted(zip(sample_titles, scores), key=lambda x: -x[1]):
    bar = '█' * int(score * 20) + '░' * (20 - int(score * 20))
    label = 'VIRAL' if score >= 0.67 else ('rising' if score >= 0.40 else 'low')
    print(f'  {bar}  {score:.2f}  [{label}]  {title}')

## 9. Save models

In [ ]:
import json as _json

# Save to Drive
joblib.dump(sentiment_pipeline, DRIVE_MODEL_DIR / 'sentiment_clf.pkl')
joblib.dump(viral_clf,          DRIVE_MODEL_DIR / 'viral_clf.pkl')
joblib.dump(title_tfidf_vec,    DRIVE_MODEL_DIR / 'viral_tfidf.pkl')

# Save metadata
meta = {
    'trained_on':        pd.Timestamp.now().isoformat(),
    'n_posts':           len(df_posts),
    'n_comments':        len(df_comments),
    'viral_threshold':   float(threshold),
    'sentiment_classes': list(sentiment_pipeline.named_steps['clf'].classes_),
    'source_files':      json_paths,
}
with open(DRIVE_MODEL_DIR / 'meta.json', 'w') as f:
    _json.dump(meta, f, indent=2)

print('Saved to Drive:')
for p in sorted(DRIVE_MODEL_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:<30} {size_kb:.1f} KB')

In [ ]:
# Also download directly to your browser
from google.colab import files

for fname in ['sentiment_clf.pkl', 'viral_clf.pkl', 'viral_tfidf.pkl', 'meta.json']:
    files.download(str(DRIVE_MODEL_DIR / fname))

print('Downloaded. Place these files in your local models/ folder.')
print('The scraper will detect and use them automatically.')

## 10. Retrain with more data (optional)

Run this cell to scrape additional posts directly from Colab (JSON API only, no browser needed)
and merge with the already-loaded data before retraining.

In [ ]:
import requests

_UA = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0.0.0 Safari/537.36'

def scrape_subreddit(sub, sort='hot', limit=100):
    posts = []
    after = None
    while len(posts) < limit:
        batch = min(limit - len(posts), 100)
        url   = f'https://www.reddit.com/r/{sub}/{sort}.json?limit={batch}'
        if after:
            url += f'&after={after}'
        r    = requests.get(url, headers={'User-Agent': _UA}, timeout=20)
        data = r.json()['data']
        for c in data['children']:
            if c.get('kind') != 't3':
                continue
            p = c['data']
            posts.append({
                'id':           p.get('name', ''),
                'title':        p.get('title', ''),
                'score':        p.get('score', 0),
                'upvote_ratio': p.get('upvote_ratio', 0.5),
                'comment_count':p.get('num_comments', 0),
                'post_type':    'self' if p.get('is_self') else 'link',
                'domain':       p.get('domain', ''),
                'subreddit':    p.get('subreddit_name_prefixed', ''),
                'created':      '',
            })
            if len(posts) >= limit:
                break
        after = data.get('after')
        if not after:
            break
        time.sleep(1)
    return pd.DataFrame(posts)


# Scrape 200 posts per sub from top (most viral ground truth)
EXTRA_SUBS  = ['MMA', 'ufc', 'boxing', 'bjj', 'wrestling']
EXTRA_SORTS = ['hot', 'top']

extra_frames = []
for sub in EXTRA_SUBS:
    for sort in EXTRA_SORTS:
        print(f'Scraping r/{sub} ({sort})...')
        extra_frames.append(scrape_subreddit(sub, sort=sort, limit=100))
        time.sleep(2)

df_extra = pd.concat(extra_frames).drop_duplicates('id')

# Merge and deduplicate
df_posts_all = pd.concat([df_posts, df_extra]).drop_duplicates('id').reset_index(drop=True)
print(f'\nTotal posts (original + extra): {len(df_posts_all)}')

# Re-label and retrain (run cells 6 again with df_posts = df_posts_all)
df_posts = df_posts_all
print('df_posts updated. Re-run cells in section 6 to retrain the viral model.')